# Install dependencies

In [26]:
!pip install pygeohash category_encoders lightgbm xgboost optuna scikit-learn pandas numpy --quiet

In [27]:
import pandas as pd
import numpy as np
import pygeohash as pgh
import warnings
import re
import optuna
import lightgbm as lgb

from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score
from category_encoders import TargetEncoder

import matplotlib.pyplot as plt

# Load data : google drive
change this cell for own data loading

In [28]:
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/gridlock-2.0/datasets"
train = pd.read_csv(f"{path}/train.csv")
test  = pd.read_csv(f"{path}/test.csv")

print(f"Train: {train.shape}   Test: {test.shape}")
train.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: (77299, 11)   Test: (41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


# Data analysis

In [29]:
print("--- dtypes ---")
print(train.dtypes)
print("\n--- missing % ---")
print((train.isnull().mean() * 100).round(2))
print("\n--- target stats ---")
print(train['demand'].describe())

--- dtypes ---
Index              int64
geohash           object
day                int64
timestamp         object
demand           float64
RoadType          object
NumberofLanes      int64
LargeVehicles     object
Landmarks         object
Temperature      float64
Weather           object
dtype: object

--- missing % ---
Index            0.00
geohash          0.00
day              0.00
timestamp        0.00
demand           0.00
RoadType         0.78
NumberofLanes    0.00
LargeVehicles    0.00
Landmarks        0.00
Temperature      3.23
Weather          1.03
dtype: float64

--- target stats ---
count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64


In [30]:
print(train['Weather'].value_counts())
print(train['RoadType'].value_counts())
print(train['NumberofLanes'].value_counts())
print(train['LargeVehicles'].value_counts())
print(train['Landmarks'].value_counts())

Weather
Sunny    27717
Rainy    20824
Foggy    20243
Snowy     7718
Name: count, dtype: int64
RoadType
Residential    69230
Street          3909
Highway         3560
Name: count, dtype: int64
NumberofLanes
1    27411
2    24127
3    23919
4      926
5      916
Name: count, dtype: int64
LargeVehicles
Not Allowed    50673
Allowed        26626
Name: count, dtype: int64
Landmarks
Yes    52042
No     25257
Name: count, dtype: int64


In [ ]:
avg_temp_by_day = train.groupby('day')['Temperature'].mean()
print("Average Temperature by Day:")
print(avg_temp_by_day)

# Feature Engineering

In [31]:
# -------------------------Timestamp-----------------------------
class TimestampEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self

    def transform(self, X):
        # X = X.copy()
        ts = pd.to_datetime(X['timestamp'], format='%H:%M', errors='coerce')
        X['hour']   = ts.dt.hour
        X['minute'] = ts.dt.minute

        # minutes passed overall
        X['overall_minutes']=X['day']*24*60+X['hour']*60+X['minute']

        # minutes passed day wise
        X['day_minutes']=X['hour']*60+X['minute']

        # cyclic day_minutes
        X['day_minutes_sin'] = np.sin(2 * np.pi * X['day_minutes'] / (24 * 60))
        X['day_minutes_cos'] = np.cos(2 * np.pi * X['day_minutes'] / (24 * 60))

        # 15-min time slot in a day (0-95)
        X['time_slot'] = X['hour'] * 4 + (X['minute'] // 15)

        # Peak-hour flags
        X['is_morning_peak'] = ((X['hour'] >= 7)  & (X['hour'] <= 9)).astype(int)
        X['is_evening_peak'] = ((X['hour'] >= 17) & (X['hour'] <= 19)).astype(int)
        X['is_night']        = ((X['hour'] >= 22) | (X['hour'] <= 5)).astype(int)

        # Interaction: day × time_slot
        X['day_time_slot'] = X['day'].astype(str) + '_' + X['time_slot'].astype(str)

        return X.drop(columns=['timestamp', 'minute'])


# --------------------------geohash---------------------------------
class GeohashEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Cache lat/lon decode to avoid repeat calls
        unique_gh = X['geohash'].dropna().unique()
        self._latlon = {gh: pgh.decode(gh) for gh in unique_gh}
        return self

    def transform(self, X):
        X = X.copy()

        X['lat'] = X['geohash'].map(lambda x: self._latlon.get(x, (np.nan, np.nan))[0])
        X['lon'] = X['geohash'].map(lambda x: self._latlon.get(x, (np.nan, np.nan))[1])

        # Hierarchical geohash prefixes
        X['geo_l6'] = X['geohash'].str[:6]
        X['geo_l5'] = X['geohash'].str[:5]
        X['geo_l4'] = X['geohash'].str[:4]
        X['geo_l3'] = X['geohash'].str[:3]

        # Interaction keys (geo5 × time features)
        X['geo5_time_slot'] = X['geo_l5'] + '_' + X['time_slot'].astype(str)
        X['geo5_hour']      = X['geo_l5'] + '_' + X['hour'].astype(str)
        X['geo5_day']       = X['geo_l5'] + '_' + X['day'].astype(str)

        # Drop raw geohash & helper cols not needed downstream
        drop_cols = ['geohash', 'geo_l6']
        return X.drop(columns=drop_cols)


# ----------------------------hierarchical temperature imputation--------------------
class HierarchicalTempImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.med_ = {}    # dict of {key_col: {key_val: median}}
        self.global_med_ = None

    def fit(self, X, y=None):
        temp     = X.iloc[:, 0].copy()
        key_cols = X.columns[1:].tolist()   # ['geo5_time_slot', 'geo5_hour', 'geo5_day']
        filled   = temp.copy()

        for kc in key_cols:
            keys = X[kc]
            med  = filled.groupby(keys).median()
            self.med_[kc] = med.to_dict()
            filled = filled.fillna(keys.map(self.med_[kc]))

        self.global_med_ = temp.median()
        return self

    def transform(self, X):
        X    = X.copy()
        temp = X.iloc[:, 0].copy()
        key_cols = X.columns[1:].tolist()

        for kc in key_cols:
            temp = temp.fillna(X[kc].map(self.med_[kc]))

        temp = temp.fillna(self.global_med_)
        return temp.to_numpy().reshape(-1, 1)


# Colum transformer

In [32]:
TEMP_COLS   = ['Temperature', 'geo5_time_slot', 'geo5_hour', 'geo5_day']
CAT_LOW     = ['RoadType', 'Weather']
CAT_LARGE   = ['LargeVehicles','Landmarks']
CAT_HIGH    = ['geo5_time_slot', 'geo5_hour', 'geo5_day',
                'geo_l5', 'geo_l4', 'geo_l3', 'day_time_slot']

col_transformer = ColumnTransformer(
    transformers=[
        # --- Temperature: hierarchical impute → RobustScaler ---
        ('temp', Pipeline([
            ('impute', HierarchicalTempImputer()),
            ('scale',  RobustScaler())
        ]), TEMP_COLS),

        # --- Low-cardinality categoricals: OHE ---
        ('cat_low', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('ohe',    OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
        ]), CAT_LOW),

        # --- Binary ordinal: LargeVehicles ---
        ('large', OrdinalEncoder(
            categories=[['Not Allowed', 'Allowed'],['No', 'Yes']],
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ), CAT_LARGE),

        # --- High-cardinality: TargetEncoder with smoothing ---
        ('cat_high', TargetEncoder(smoothing=15, min_samples_leaf=10),
            CAT_HIGH),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
    n_jobs=-1
)


# Preprocess pipeline

In [33]:
preproc_pipeline = Pipeline(steps=[
    ('ts_eng',  TimestampEngineer()),
    ('geo_eng', GeohashEngineer()),
    ('col_tr',  col_transformer),
])

print('Preprocessing pipeline defined.')

Preprocessing pipeline defined.


# Train test split , target transform

In [38]:
# Drop unwanted colums
DROP_RAW = ['Index']

X = train.drop(columns=DROP_RAW + ['demand'])
y = train['demand'].copy()

X_test_raw = test.drop(columns=[c for c in DROP_RAW if c in test.columns])

# Log1p transform on target — demand is likely right-skewed
y_log = np.log1p(y)

print(f"X: {X.shape}   y skew before: {y.skew():.2f}  after log1p: {y_log.skew():.2f}")
print("Columns:", X.columns.tolist())

X: (77299, 9)   y skew before: 3.73  after log1p: 2.97
Columns: ['geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']


# hyperparameter tuning using optuna

In [39]:
N_TRIALS  = 100
N_FOLDS_TUNE  = 3
RANDOM_SEED   = 42

def objective(trial):
    params = {
        'n_estimators':       trial.suggest_int('n_estimators', 500, 3000, step=100),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves':         trial.suggest_int('num_leaves', 31, 255),
        'max_depth':          trial.suggest_int('max_depth', 4, 12),
        'min_child_samples':  trial.suggest_int('min_child_samples', 10, 100),
        'subsample':          trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_split_gain':     trial.suggest_float('min_split_gain', 0.0, 0.5),
        'random_state':       RANDOM_SEED,
        'verbose':            -1,
        'n_jobs':             -1,
    }

    kf    = KFold(n_splits=N_FOLDS_TUNE, shuffle=True, random_state=RANDOM_SEED)
    r2_scores = []

    for tr_idx, va_idx in kf.split(X):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y_log.iloc[tr_idx], y_log.iloc[va_idx]

        # Fit preprocessing on train fold only (avoids leakage)
        pp = preproc_pipeline

        X_tr_t = pp.fit_transform(X_tr, y_tr)
        X_va_t = pp.transform(X_va)

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr_t, y_tr,
            eval_set=[(X_va_t, y_va)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        )

        preds = model.predict(X_va_t)
        r2  = r2_score(y_va, preds)
        r2_scores.append(r2)

    return np.mean(r2_scores)


study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest R2 score (log-scale): {study.best_value:.5f}")
print("Best params:", study.best_params)


  0%|          | 0/80 [00:00<?, ?it/s]


Best R2 score (log-scale): 0.90821
Best params: {'n_estimators': 2800, 'learning_rate': 0.024000508657683623, 'num_leaves': 197, 'max_depth': 10, 'min_child_samples': 23, 'subsample': 0.9153338941085418, 'colsample_bytree': 0.7763571099731742, 'reg_alpha': 0.1628327630780285, 'reg_lambda': 0.8319713754399246, 'min_split_gain': 1.7278791055352644e-05}


# final model training and predicting

In [41]:
best_params   = study.best_params
best_params.update({'random_state': RANDOM_SEED, 'verbose': -1, 'n_jobs': -1})

# Prepare data for training a single model
print("\nFitting preprocessing pipeline on full training data...")
pp = preproc_pipeline
X_train_transformed = pp.fit_transform(X, y_log)
X_test_transformed  = pp.transform(X_test_raw)

# Train a single model with best parameters on the full training data
print("Training final LightGBM model...")
model = lgb.LGBMRegressor(**best_params)
model.fit(X_train_transformed, y_log)

# Make predictions on the full training data (for OOF R2, if desired) and test data
print("Making predictions...")
oof_preds   = model.predict(X_train_transformed)
test_preds  = model.predict(X_test_transformed)

# Overall OOF score (calculated on the full training data predictions)
oof_r2 = r2_score(y_log, oof_preds)
print(f"\n★ Full Training R2 Score (log-scale): {oof_r2:.5f}")

# Inverse transform predictions to original scale
oof_demand  = np.expm1(oof_preds)
test_demand = np.expm1(test_preds)

# R2 score on original scale
orig_r2 = r2_score(y, oof_demand)
print(f"★ Full Training R2 Score (original scale):    {orig_r2:.5f}")



Fitting preprocessing pipeline on full training data...
Training final LightGBM model...
Making predictions...

★ Full Training R2 Score (log-scale): 0.95900
★ Full Training R2 Score (original scale):    0.96775


# Generate submission file
change target file path to save at that location

In [44]:
test_demand_clipped = np.clip(test_demand, 0, None)

submission = pd.DataFrame({
    'Index':  test['Index'],
    'demand': test_demand_clipped
})

submission.to_csv(f"{path}/output.csv", index=False)
print("Saved submission!")
submission.head()

Saved submission!


,Index,demand
0,0,0.046041
1,1,0.054598
2,2,0.038122
3,3,0.032916
4,4,0.058526


---
## Design Decisions & Rationale

| Decision | Old | New | Why |
|---|---|---|---|
| **Model** | XGBRegressor (default) | LGBMRegressor (tuned) | LightGBM is 3-5× faster, handles categoricals natively, typically ≥ XGB on tabular traffic data |
| **Hyperparameter search** | GridSearchCV (3 fixed combos) | Optuna Bayesian TPE (80 trials) | Explores 8+ dimensions intelligently, finds num_leaves/reg_alpha that grid search misses |
| **Target transform** | None | log1p | Demand is right-skewed; log1p reduces outlier influence and MSE sensitivity |
| **Temperature imputation** | Hierarchical median | Same + RobustScaler | RobustScaler is better than StandardScaler for distributions with outliers |
| **Cyclic encoding** | hour + day only | + minute sin/cos | 15-min granularity matters for traffic; minute=0,15,30,45 all behave differently |
| **Peak flags** | None | morning_peak, evening_peak, night, weekend | Direct domain knowledge; huge SHAP impact on demand |
| **Geohash levels** | l3–l6 (dropped all) | l3–l5 kept for target encoding | Coarser geohash captures regional trends; finer captures local patterns |
| **Weather** | Dropped | Cleaned → 6 buckets → OHE | Rain/Storm meaningfully elevates demand; dropping it wastes signal |
| **Landmarks** | Dropped | has_landmark (0/1) + landmark_count | Proximity to landmarks raises demand |
| **Target encoding** | smoothing=10 | smoothing=15, min_samples_leaf=10 | Prevents overfitting on rare geo×time combos |
| **Cross-validation** | 5-fold GridSearchCV | 5-fold OOF KFold + Optuna 3-fold | OOF prevents leakage; gives honest RMSE before submission |
| **Geohash l4 interaction** | Missing | geo4_time_slot | Adds an intermediate spatial resolution that improves preds |
